In [ ]:
%load_ext autoreload
%autoreload 2
%cd ../../..
from rich import print


# Configuration

Edit the cells below to set the case, sector avoidance, flight selection, and run parameters.

In [ ]:
# ===== Case & model =====
case_dir = "data/cases/LGAV_LFPG/"
batch_sgd_results_dir_path = f"{case_dir}results_full/"

# ===== Sector avoidance =====
sectors_to_avoid = ["LFEEE"]          # list of sector IDs to exclude
sectors_geojson_path = "data/airspace/sectors.geojson"
connectivity_repair_iterations = 20   # post-sector connectivity repair iterations

# ===== Flight selection =====
reference_flight_id = "46B8A6AEE4CA"
reference_takeoff_timestamp = 1681225713

# ===== Run settings =====
run_name = None                        # None = auto timestamp-based name
n_samples = 100
policy = "sample"                      # "sample" or "greedy"
initial_k_policy = "uniform"           # "uniform" or "latest"
max_flight_duration_hours = 3.0 + 20.0 / 60.0
use_flight_metadata_duration = True
seed = 7


In [ ]:
# Look for the latest checkpoint in the `batch_sgd_results_dir_path`
import re
from pathlib import Path

def get_latest_checkpoint(results_dir_path):
    directory = Path(results_dir_path)
    
    # Pattern to match 'checkpoint_iter_X.pt' and capture X
    pattern = re.compile(r"checkpoint_iter_(\d+)\.pt")
    
    checkpoints = []
    
    # Iterate through files in the directory
    for file in directory.glob("checkpoint_iter_*.pt"):
        match = pattern.search(file.name)
        if match:
            iteration = int(match.group(1))
            checkpoints.append((iteration, file.name))
            
    if not checkpoints:
        return None
    
    # Sort by the iteration number (the first element of the tuple) and get the last one
    latest_checkpoint = max(checkpoints, key=lambda x: x[0])
    
    return latest_checkpoint[1]


latest_checkpoint_filename = batch_sgd_results_dir_path + get_latest_checkpoint(batch_sgd_results_dir_path)
print(f"Latest checkpoint: {latest_checkpoint_filename}")

In [ ]:
# Derive the run directory and output/cache directories
from datetime import datetime
from pathlib import Path

_run_name = run_name or datetime.now().strftime("run_%Y%m%d_%H%M%S")
_sectors_tag = "_".join(sectors_to_avoid) if sectors_to_avoid else "no_sectors"
run_dir = Path(case_dir) / "err41" / f"{_run_name}_avoid_{_sectors_tag}"
output_dir = run_dir / "inference_outputs"
cache_dir = run_dir / "inference_cache"
run_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)
print(f"Run dir:    {run_dir}")
print(f"Output dir: {output_dir}")
print(f"Cache dir:  {cache_dir}")


# Learned Edge Preferences (Full Graph)

Inspect the checkpoint parameters and visualize preferences on the unmodified route graph.

In [ ]:
from equinox.posttrain import load_cost_model_parameters
from rich.console import Console
from rich.table import Table

def print_params_table(params_dict):
    console = Console()
    
    # Initialize the table
    table = Table(title="Common Parameters", show_header=True, header_style="bold magenta")
    
    table.add_column("Parameter", style="dim", width=12)
    table.add_column("Value", justify="right")

    # Add rows, formatting floats to 6 decimal places for clarity
    for key, value in params_dict.items():
        table.add_row(key, f"{value:.6f}")

    console.print(table)

params = load_cost_model_parameters(latest_checkpoint_filename)
print("[bold red]Parameters retrieved from learned cost model[/bold red]")
print_params_table(params["common"])
print(params["preference_matrix"].shape)  # (num_nodes, num_nodes)

In [ ]:
from equinox.posttrain import load_checkpoint_edge_preferences, plot_edge_preferences_cartopy

from pathlib import Path

def get_gml_path(batch_sgd_results_dir_path: str) -> str:
    """
    Recursively searches for a .gml file in the specified directory.
    Returns the absolute path to the .gml file.
    """
    directory = Path(batch_sgd_results_dir_path)
    
    # rglob("*.gml") searches recursively for all files ending in .gml
    # We use next() to get the first result since only one is guaranteed.
    try:
        gml_file = next(directory.rglob("*.gml"))
        return str(gml_file.absolute())
    except StopIteration:
        raise FileNotFoundError(f"No .gml file found in {batch_sgd_results_dir_path}")

gml_path = get_gml_path(case_dir)
print(f'Loading route graph from {gml_path}')

payload = load_checkpoint_edge_preferences(latest_checkpoint_filename, gml_path)
plot_edge_preferences_cartopy(
    payload["graph"],
    edge_preferences=payload["edge_preferences"],
    cmap="coolwarm",
    linewidth=2,
    alpha=0.4,
    percentile=0.5,
    percentile_mode="contrast", # upper = more preferred, lower = less preferred, contrast = half lowest, half highest
    preference_color_range=(-0.5,0.5)
)

In [ ]:
plot_edge_preferences_cartopy(
    payload["graph"],
    edge_preferences=payload["edge_preferences"],
    cmap="coolwarm",
    linewidth=4,
    alpha=0.9,
    percentile=0.01,
    percentile_mode="contrast", # upper = more preferred, lower = less preferred, contrast = half lowest, half highest
    preference_color_range=(-0.5,0.5),
    show_waypoints=True
)

# Raw Prior Route Traversal Frequency

In [ ]:
from equinox.posttrain.plot_sample_routes import plot_snapped_routes
plot_snapped_routes(case_dir=Path(case_dir), show_waypoints=False)

# Airspace Avoidance: Graph After Sector Severing

Apply `GraphScenario` edge-filter mode to remove graph edges that cross the specified sectors.  
Then visualize the surviving graph with learned preferences to understand reachability after avoidance.

In [ ]:
from equinox.sampling.pipeline import GraphScenario, _prepare_graph_for_scenario
from equinox.sampling.trespass.inference import load_case

graph_scenario = GraphScenario(
    mode="edge_filter",
    sectors_to_avoid=sectors_to_avoid,
    sectors_geojson_path=sectors_geojson_path,
    post_sector_connectivity_repair=False
    # connectivity_repair_iterations=connectivity_repair_iterations,
)

# Load base components and apply graph scenario
config, base_components = load_case(case_dir)
severed_components, graph_report = _prepare_graph_for_scenario(
    config=config,
    components=base_components,
    graph_scenario=graph_scenario,
)

print(f"[bold green]Graph scenario report[/bold green]")
for k, v in graph_report.items():
    if not isinstance(v, dict):
        print(f"  [cyan]{k}[/cyan]: {v}")
    else:
        print(f"  [cyan]{k}[/cyan]:")
        for kk, vv in v.items():
            print(f"    {kk}: {vv}")


In [ ]:
# Visualize the graph after sector edge severing.
# Filter the learned preferences to only the surviving edges.
severed_graph = severed_components["graph"]
severed_edge_set = set(severed_graph.edges())
filtered_prefs = {
    e: v for e, v in payload["edge_preferences"].items()
    if e in severed_edge_set
}

print(f"Original edges:          {payload['graph'].number_of_edges()}")
print(f"Edges after severing:    {severed_graph.number_of_edges()}")
print(f"Preferences retained:    {len(filtered_prefs)}")
print(f"Sectors avoided:         {sectors_to_avoid}")

plot_edge_preferences_cartopy(
    severed_graph,
    edge_preferences=filtered_prefs,
    cmap="coolwarm",
    linewidth=2,
    alpha=0.6,
    percentile=0.5,
    percentile_mode="contrast",
    preference_color_range=(-0.5, 0.5),
    filter_non_clsr=False,
    show_waypoints=True,
)


In [ ]:
# Same view without waypoints for cleaner edge density reading
plot_edge_preferences_cartopy(
    severed_graph,
    edge_preferences=filtered_prefs,
    cmap="coolwarm",
    linewidth=3,
    alpha=0.8,
    percentile=0.01,
    percentile_mode="contrast",
    preference_color_range=(-0.5, 0.5),
    filter_non_clsr=False,
    show_waypoints=False,
)


# Route Cost Decomposition and Inspection

In [ ]:
from equinox.posttrain.cost_back_envelope import compute_route_cost_breakdown, breakdown_to_frame

result = compute_route_cost_breakdown(
    checkpoint_path=latest_checkpoint_filename,
    case_dir=case_dir,
    flight_id=reference_flight_id
    # takeoff_time=reference_takeoff_timestamp
)

df = breakdown_to_frame(result["breakdown"])
print_params_table(result["totals"])
df


# Inference / Sampling (Sector-Avoidance Run)

Run the backward SVI pipeline with the sector-filtered graph.  
Outputs are isolated under `<case_dir>/err41/<run_name>_avoid_<sectors>/` to avoid collisions with other runs.

In [ ]:
from datetime import datetime, timedelta

from rich.console import Console
from rich.table import Table

from equinox.sampling.pipeline import _resolve_takeoff_fields
from equinox.sampling.trespass.inference import load_flight_artifacts

# Resolve the selected flight and its raw metadata
(
    _selected_flight_id,
    selected_takeoff_ts,
    _transitions,
    _tailwind,
    _flight_metadata,
) = load_flight_artifacts(
    str(case_dir),
    flight_id=reference_flight_id,
    takeoff_timestamp=reference_takeoff_timestamp,
)

flight_meta = _flight_metadata.get("flight_metadata", {}) if isinstance(_flight_metadata, dict) else {}

# Human-readable takeoff string
_, takeoff_time_str = _resolve_takeoff_fields(takeoff_timestamp=int(selected_takeoff_ts))

# Try to infer flight duration from metadata (same logic as err41_full_inference.py)
inferred_duration_hours = None
try:
    ft = flight_meta.get("flight_time_s")
    if ft is not None:
        inferred_duration_hours = float(ft) / 3600.0
except (TypeError, ValueError):
    pass

if inferred_duration_hours is None:
    try:
        t0 = flight_meta.get("takeoff_time")
        t1 = flight_meta.get("landing_time")
        if t0 is not None and t1 is not None:
            inferred_duration_hours = max(0.0, (float(t1) - float(t0)) / 3600.0)
    except (TypeError, ValueError):
        pass

# Determine effective duration
if use_flight_metadata_duration and inferred_duration_hours is not None and inferred_duration_hours > 0:
    effective_duration_hours = inferred_duration_hours
    duration_source = "flight_metadata"
else:
    effective_duration_hours = max_flight_duration_hours
    duration_source = "fixed_max_flight_duration_hours"

# Derive estimated landing time
_takeoff_dt = datetime.fromtimestamp(int(selected_takeoff_ts))
_landing_dt = _takeoff_dt + timedelta(hours=effective_duration_hours)
estimated_landing_time_str = _landing_dt.strftime("%Y-%m-%d %H:%M:%S")

# Print summary table
console = Console()
table = Table(title="Resolved Flight & Inference Metadata", show_header=True, header_style="bold magenta")
table.add_column("Field", style="dim", min_width=32)
table.add_column("Value", justify="right")

table.add_row("flight_id", str(_selected_flight_id))
table.add_row("takeoff_timestamp", str(int(selected_takeoff_ts)))
table.add_row("takeoff_time_str", takeoff_time_str)
table.add_row("inferred_duration_hours (metadata)",
              f"{inferred_duration_hours:.4f} h" if inferred_duration_hours is not None else "N/A")
table.add_row("max_flight_duration_hours (config)", f"{max_flight_duration_hours:.4f} h")
table.add_row("effective_duration_hours", f"{effective_duration_hours:.4f} h")
table.add_row("duration_source", duration_source)
table.add_row("estimated_landing_time_str", estimated_landing_time_str)
table.add_row("sectors_to_avoid", str(sectors_to_avoid))
table.add_row("n_samples", str(n_samples))
table.add_row("policy", policy)
table.add_row("initial_k_policy", initial_k_policy)
table.add_row("output_dir", str(output_dir))

console.print(table)


In [ ]:
from equinox.sampling.pipeline import compute_4d_path_for_dataset
from equinox.posttrain.plot_sample_routes import plot_sample_routes

res = compute_4d_path_for_dataset(
    case_dir=str(case_dir),
    checkpoint_path=str(latest_checkpoint_filename),
    flight_id=reference_flight_id,
    takeoff_timestamp=reference_takeoff_timestamp,
    n_samples=n_samples,
    policy=policy,
    initial_k_policy=initial_k_policy,
    seed=seed,
    output_dir=str(output_dir),
    cache_dir=str(cache_dir),
    use_cache=False,
    write_4d_csv=True,
    graph_scenario=graph_scenario,
    transition_mode="recompute",
    estimated_landing_time_str=estimated_landing_time_str,
)

print(f"Sampled {len(res.samples)} routes")
if res.samples:
    print(f"First route: {res.samples[0].route}")


In [ ]:
plot_sample_routes(
    case_dir=Path(case_dir),
    routes_path=output_dir / "routes.txt",
    show=True,
    show_waypoints=False,
    route_alpha=0.01,
    reference_flight_id=reference_flight_id,
    reference_takeoff_timestamp=reference_takeoff_timestamp,
    plot_wind=True,
    show_quiver=True,
    thickness=10,
    plot_title=f"Inference for Flight {reference_flight_id} (Avoiding: {sectors_to_avoid})"
)


In [ ]:
from equinox.posttrain.cost_back_envelope import compute_routes_cost_breakdowns

compute_routes_cost_breakdowns(
    checkpoint_path=latest_checkpoint_filename,
    case_dir=case_dir,
    flight_id=reference_flight_id,
    takeoff_time=reference_takeoff_timestamp,
    routes_path=output_dir / "routes.txt",
    limit=10
)
